# v2.03beta — step 0 of the v2.04 plan, on Linux (Colab)

**Runtime:** any GPU (T4, L4 or A100), then Runtime → Run all. About **15–25 min** in total: tests ~1 min, 2026 data ~5 min, v2.03 predictions ~5–10 min, XGBoost ~5 min, scoring ~2 min.

**Why Colab and not the laptop (2026-10-27).** The ADX input depends on the platform on "tie" days, when a stock's up-move equals its down-move (`claude/log-entry-2026-10-27-adx-tie-platform.md`). v2.03 was trained on Colab (Linux), so its inputs are reproduced exactly only on Linux. The Windows run failed the reproduction check and is not used.

**What it does** (`scripts/step0_v2_03beta.py`, design `claude/log-entry-2026-10-26-v2-03beta-design.md`). There is no re-training: the three seeds' saved v2.03 weights are only used for predictions.
- **Methods:** temperature scaling, bias-corrected temperature scaling, the per-date ⅓ rank call-mix, per-date class-share EM, and the deep + XGBoost blend (XGBoost trained on the same fit rows). Each is fitted on the inner hold-out only.
- **News placebo:** 5 shuffles, on validation and on 2026, plus news blanked.
- **Scoring:** validation decides. 2026 is a seen window and is only reported. The 2025 test block stays sealed.

**Stops** if the code differs from v2.03's, if a 2026 input file differs from the forward tests' copy, if the identity check fails, or if the predictions here do not reproduce v2.03's stored validation predictions (within 0.001) and each seed's inner hold-out F1 (within 0.002).

Outputs go to Drive `Senior/results/v2_03beta/`. Every stage is resumable: re-running the notebook skips the parts already saved.

In [ ]:
!nvidia-smi -L
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU works, but predictions take hours)')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, subprocess, sys
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, RES = f'{SENIOR}/prices', f'{SENIOR}/results'
RUN_JSON = f'{RES}/v2xx/A100/v2_03.json'                 # v2.03 (A100, 2026-10-24) and its weights folder beside it
FWD_INPUTS = f'{RES}/forward_2026/inputs'                # the saved 2026 prices and events of the forward tests
OUT = f'{RES}/v2_03beta'                                 # outputs of this notebook
RUN = json.load(open(RUN_JSON))
SEEDS = [r['seed'] for r in RUN['runs']]
need = [RUN_JSON, f'{RES}/v2xx/A100/v2_03_preds.csv.gz', f'{FWD_INPUTS}/forward_prices.csv', f'{FWD_INPUTS}/forward_events.csv',
        f'{FWD_INPUTS}/inputs_meta.json', f'{PRICE_DIR}/set50_union_2019_2025.csv', f'{PRICE_DIR}/yahoo_events_2019_2025.csv']
need += [f'{RES}/v2xx/A100/v2_03_weights/seed{s}.pt' for s in SEEDS]
for f in need:
    assert os.path.exists(f), f'stop: missing {f}'
os.makedirs(OUT, exist_ok=True)
print('v2.03 run:', RUN['git_commit'], RUN['gpu'], 'seeds', SEEDS, '| outputs go to', OUT)

def run(args):
    # stream a script's output into the notebook and stop on failure
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if 'defined on' not in line and 'not writable' not in line and 'from_numpy' not in line:
            print(line, end='', flush=True)
    if p.wait() != 0:
        raise SystemExit(f'stop: {" ".join(args[1:3])} failed (exit {p.returncode}); see the lines above')

In [ ]:
# Latest code (step 0 is new; the v2.03 model files must still be the ones v2.03 was trained with)
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
assert os.path.exists('scripts/step0_v2_03beta.py'), 'stop: scripts/step0_v2_03beta.py is not on GitHub yet (commit and push it)'
for k, f in [('script_sha256', 'scripts/train_v2xx.py'), ('model_sha256', 'src/proposal_model.py'),
             ('encoders_sha256', 'src/encoders.py'), ('features_sha256', 'src/features_proposal.py')]:
    g = hashlib.sha256(open(f, 'rb').read()).hexdigest()
    print('OK ' if g == RUN[k] else 'MISMATCH ', f)
    assert g == RUN[k], f'stop: {f} is not the code v2.03 was trained with'
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    g = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if g == h else 'MISMATCH ', path)
    assert g == h, 'stop: a file differs from the frozen copy'
import platform, numpy, pandas
print('platform', platform.platform(), '| numpy', numpy.__version__, '| pandas', pandas.__version__, '| torch', torch.__version__)

In [ ]:
# Tests of the step-0 methods (about 1 minute)
r = subprocess.run(['python', 'tests/test_step0_v2_03beta.py'], capture_output=True, text=True)
print(r.stdout[-2500:])
assert r.returncode == 0 and 'FAIL' not in r.stdout, 'stop: tests failed\n' + r.stderr[-2000:]

In [ ]:
# 1. 2026 data (the forward tests' saved inputs) and the identity check (training and validation rows unchanged)
common = ['--run-json', RUN_JSON, '--fwd-inputs', FWD_INPUTS, '--out-dir', OUT]
run(['python', 'scripts/step0_v2_03beta.py', 'prepare', *common])

In [ ]:
# 2. v2.03 predictions from the saved weights: inner hold-out, validation, 2026, news blanked, 5 placebo shuffles x 2 periods
run(['python', 'scripts/step0_v2_03beta.py', 'predict', *common])

In [ ]:
# 3. XGBoost on the same fit rows as v2.03 (3 seeds x 3 horizons)
run(['python', 'scripts/step0_v2_03beta.py', 'trees', *common])

In [ ]:
# 4. Reproduction check, then every method and the placebo (prints the tables; writes step0.json)
run(['python', 'scripts/step0_v2_03beta.py', 'analyze', *common])

In [ ]:
# 5. Compact view of step0.json
import pandas as pd
from IPython.display import display
s = json.load(open(f'{OUT}/step0.json'))
print('reproduction ok:', s['reproduction']['ok'], '| call-mix chosen on the inner hold-out:', s['choices']['call_mix'],
      '| blend w (deep):', {h: v['w_deep'] for h, v in s['choices']['blend_w'].items()})
rows = []
for k, a in s['adopt'].items():
    r = {'method': k, 'adopted': a['adopted']}
    for h in ('h1', 'h5', 'h10'):
        v = s['validation'][h][k]
        r[f'{h} F1'] = v['macro_f1']; r[f'{h} vs raw'] = v.get('minus_raw', {}).get('point')
    rows.append(r)
display(pd.DataFrame(rows).set_index('method'))
print('inner hold-out (cross-fitted), 3-horizon mean:', s['inner_holdout_cross_fitted']['three_horizon_mean'])
for per, out in s['news_placebo'].items():
    print(f'\nnews placebo, {per}:')
    display(pd.DataFrame({h: {k: out[h][k]['macro_f1'] for k in ('news_on', 'placebo', 'blanked')}
                          | {'news_on - placebo': out[h]['differences']['news_on - placebo']['point'],
                             'interval': out[h]['differences']['news_on - placebo']['ci95']} for h in ('h1', 'h5', 'h10')}))

**Reading the result**
- **Adopted** follows the pre-registered rule. A call-changing method is adopted if its cross-fitted inner hold-out score is above raw and its validation macro-F1 is not below raw at any horizon. Temperature scaling is adopted if it lowers validation log-loss at every horizon (its calls never change).
- **vs raw** is the change on the same validation rows. Its 95% date-block interval is in `step0.json` and in the printed table above (`*` = interval excludes 0).
- **News placebo:** if *news on − placebo* is about 0 while the placebo beats *blanked*, what helps is which days have news, not what the news says. If *news on − placebo* > 0, the content matters.
- **2026** is a seen window: printed for reference, never used to adopt.
- Send the printout to Claude when it finishes.